# The KDE defence — and the KDE term turned back into an attack

Kernel density estimation appears in this project on both sides of the fight, and this notebook keeps the whole story in one place. **Part 1** builds the KDE function itself and studies its bandwidths, kernels and centres. **Part 2** uses it as a **defence**: a detector that flags inputs lying in low-density regions, measured against the σ attack. **Part 3** flips it around — the GKDE attack adds a KDE term to PGD so the adversarial examples stay in high-density (benign-looking) regions, with the parameter grid the paper reports.

**Where this fits:** Section 03 built the attacks; this section asks how the **defences** hold up against them — above all against the σ attack that grew out of `03-attacks/cw_sine_lineage.ipynb`. The same defences, measured in the final organised protocol, are `06-thesis-evaluation/`.

*Consolidated from `KDE0.ipynb`, `KDE.ipynb` (both originally in `PGD_form_attacks` / `mal_adv3`) and `kde1.ipynb` (originally in `mal_adv3`). The wider GKDE study is in `03-attacks/pgd_analysis.ipynb`. Outputs cleared.*

## Part 1 — The KDE function itself

### Step 1 — KDE in NumPy/sklearn: bandwidth and kernel choices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Step 1: Generate the dataset
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', s=20)
plt.title("Decision Boundary with Gaussian Kernel Density Estimation")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
def KDE(x, centers, bandwidth,kernel):

    n_samples, n_features = X.shape
    n_train_samples = centers.shape[0]
    u = np.abs(x[:, None, :] - centers)

    if kernel == 'gaussian':
        scores = (np.exp(-0.5 * np.sum(u**2, axis=-1) / bandwidth**2)).mean(axis=1)
        normalization_constant = (2 * np.pi * bandwidth**2) ** (n_features / 2)
    else :
        scores = (np.exp(-np.sum(u, axis=-1) / bandwidth)).mean(axis=1)
        normalization_constant = (2 * bandwidth) ** n_features



    return scores / normalization_constant

In [ ]:
bandwidth = 0.1
kernel = 'gaussian'

# Compute density scores for the test set
dens_benign = KDE(X_test, X_train[y_train == 0], bandwidth,kernel)
dens_malware = KDE(X_test, X_train[y_train == 1], bandwidth,kernel)

# Step 3: Classify samples based on KDE
y_pred = np.where(dens_benign > dens_malware, 0, 1)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Plot the dataset and decision boundary
xx, yy = np.meshgrid(np.linspace(-2, 3, 500), np.linspace(-1, 2, 500))
grid = np.c_[xx.ravel(), yy.ravel()]

dens_benign_grid = KDE(grid, X_train[y_train == 0], bandwidth,kernel)
dens_malware_grid = KDE(grid, X_train[y_train == 1], bandwidth,kernel)

decision = dens_benign_grid > dens_malware_grid
decision = decision.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, decision, cmap='coolwarm', alpha=0.8)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=20)
plt.title("Decision Boundary with Gaussian Kernel Density Estimation")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
bens = X_test[y_test==0]
mals = X_test[y_test==1]
print(KDE(bens[:10], X_train[y_train == 0], bandwidth,kernel))
print(KDE(mals[:10], X_train[y_train == 0], bandwidth,kernel))

In [ ]:
print(KDE(bens[:10], X_train[y_train == 1], bandwidth,kernel))
print(KDE(mals[:10], X_train[y_train == 1], bandwidth,kernel))

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.05
kernel = 'gaussian'

# Create a mesh grid
xx, yy = np.meshgrid(np.linspace(-2, 3, 500), np.linspace(-1, 2, 500))
grid = np.c_[xx.ravel(), yy.ravel()]

# Compute density estimates for the grid points
dens_benign_grid = KDE(grid, X_train[y_train == 0], bandwidth,kernel)
dens_benign_grid = dens_benign_grid.reshape(xx.shape)

# Plot the contour of the score for kde_benign
plt.figure(figsize=(20, 12))
plt.contourf(xx, yy, dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores for Benign Samples")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.1
kernel = 'gaussian'

# Create a mesh grid
xx, yy = np.meshgrid(np.linspace(-2, 3, 500), np.linspace(-1, 2, 500))
grid = np.c_[xx.ravel(), yy.ravel()]

# Compute density estimates for the grid points
dens_benign_grid = KDE(grid, X_train[y_train == 0], bandwidth,kernel)
dens_benign_grid = dens_benign_grid.reshape(xx.shape)

# Plot the contour of the score for kde_benign
plt.figure(figsize=(20, 12))
plt.contourf(xx, yy, dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores for Benign Samples")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.2
kernel = 'gaussian'

# Create a mesh grid
xx, yy = np.meshgrid(np.linspace(-2, 3, 500), np.linspace(-1, 2, 500))
grid = np.c_[xx.ravel(), yy.ravel()]

# Compute density estimates for the grid points
dens_benign_grid = KDE(grid, X_train[y_train == 0], bandwidth,kernel)
dens_benign_grid = dens_benign_grid.reshape(xx.shape)

# Plot the contour of the score for kde_benign
plt.figure(figsize=(20, 12))
plt.contourf(xx, yy, dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores for Benign Samples")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.4
kernel = 'gaussian'

# Create a mesh grid
xx, yy = np.meshgrid(np.linspace(-2, 3, 500), np.linspace(-1, 2, 500))
grid = np.c_[xx.ravel(), yy.ravel()]

# Compute density estimates for the grid points
dens_benign_grid = KDE(grid, X_train[y_train == 0], bandwidth,kernel)
dens_benign_grid = dens_benign_grid.reshape(xx.shape)

# Plot the contour of the score for kde_benign
plt.figure(figsize=(20, 12))
plt.contourf(xx, yy, dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores for Benign Samples")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

### Step 2 — A differentiable KDE in PyTorch, and PGD against it

#Gradients

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_losses(loss1, loss2, loss3, loss4):
    """
    Plots three lists of loss values against iterations using seaborn.

    Parameters:
    loss1 (list): First list of loss values.
    loss2 (list): Second list of loss values.
    loss3 (list): Third list of loss values.
    loss4 (list): Third list of loss values.
    """
    # Create a list of iterations based on the length of the loss lists
    iterations = list(range(1, len(loss1) + 1))

    # Create a DataFrame
    data = pd.DataFrame({
        'Iteration': iterations * 4,
        'Loss': loss1 + loss2 + loss3 + loss4,
        'Type': ['Loss1'] * len(loss1) + ['Loss2'] * len(loss2) + ['Loss3'] * len(loss3)+ ['Loss4'] * len(loss4)
    })

    # Create the plot
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=data, x='Iteration', y='Loss', hue='Type')
    plt.title('Loss vs. Iteration')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.legend(title='Loss Type')
    plt.grid(True)
    plt.show()

In [ ]:
def KDE(x, centers, bandwidth, kernel='laplacian'):
    n_samples, n_features = x.shape
    n_train_samples = centers.shape[0]
    u = torch.abs(x[:, None, :] - centers)

    if kernel == 'gaussian':
        scores = torch.exp(-0.5 * torch.sum(u**2, dim=-1) / bandwidth**2).mean(dim=1)
        normalization_constant = (2 * torch.pi * bandwidth**2) ** (n_features / 2)
    elif kernel == 'laplacian':
        scores = torch.exp(-torch.sum(u, dim=-1) / bandwidth).mean(dim=1)
        normalization_constant = (2 * bandwidth) ** n_features
    else:
        raise ValueError("Unsupported kernel. Use 'gaussian' or 'laplacian'.")

    return scores / normalization_constant

def pgd(x, centers, criterion, bandwidth, kernel, step_length=0.02, k=100, norm='linf'):
    eps = 1e-10
    x_next = x.clone()

    trajectories = [x_next.clone().detach().cpu().numpy()]
    loss_steps_i = []

    for t in range(k):
        x_var = x_next.clone().detach().requires_grad_(True)
        k = KDE(x_var, centers, bandwidth, kernel)
        if criterion == 'log':
            loss = -torch.log(k + eps)
        else:
            loss = -k

        loss_steps_i.append(k.mean().detach().item())

        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)

        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
        elif norm == 'free':
            perturbation = gradients
        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
        elif norm == 'l1':
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
        else:
            raise ValueError("Expect 'l1', 'l2', or 'linf' norm.")

        x_next = x_next + step_length * perturbation
        trajectories.append(x_next.clone().detach().cpu().numpy())

    return x_next, loss_steps_i, np.array(trajectories)

In [ ]:
# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

bens = X_test_tensor[y_test==0]
mals = X_test_tensor[y_test==1]

In [ ]:
x_next, loss_steps_i, trajectories = pgd(mals,X_train_tensor[y_train==0],'log', 0.1,'gaussian',step_length=0.01, k=100, norm='linf')
x_next2,loss_steps_i2, trajectories2 = pgd(mals,X_train_tensor[y_train==0],'linear', 0.1,'gaussian',step_length=0.01, k=100, norm='linf')

In [ ]:
KDE(mals[:20], X_train_tensor[y_train==0], 0.1,'gaussian')

In [ ]:
KDE(x_next[:20], X_train_tensor[y_train==0], 0.1,'gaussian')

In [ ]:
plot_losses(loss_steps_i, loss_steps_i2, loss_steps_i2, loss_steps_i2)

In [ ]:
x_next,loss_steps_i, trajectories = pgd(mals,X_train_tensor[y_train==0],'log', 1.,'gaussian',step_length=0.01, k=100, norm='free')
x_next2,loss_steps_i2, trajectories = pgd(mals,X_train_tensor[y_train==0],'linear', 1.,'gaussian',step_length=0.01, k=100, norm='free')

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.1
centers_benign = X_train_tensor[y_train == 0]

# Create a mesh grid
xx, yy = torch.meshgrid(torch.linspace(-2, 3, 500), torch.linspace(-1, 2, 500))
grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)

# Compute density estimates for the grid points using the improved KDE function
dens_benign_grid = KDE(grid, centers_benign, bandwidth, kernel='gaussian')
dens_benign_grid = dens_benign_grid.reshape(xx.shape).detach().numpy()

# Select a few samples to track their movement
mals = torch.tensor(X_test[y_test == 1][:10], dtype=torch.float32)

# Perform PGD
x_next, loss_steps_i, trajectories = pgd(mals, X_train_tensor[y_train == 0], 'log', bandwidth, 'gaussian', step_length=0.01, k=100, norm='linf')

# Plot the contour of the KDE score
plt.figure(figsize=(20, 12))
plt.contourf(xx.numpy(), yy.numpy(), dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')

# Plot the initial and final positions of the samples
plt.scatter(mals[:, 0], mals[:, 1], color='red', label='Initial positions')
plt.scatter(x_next[:, 0].detach().numpy(), x_next[:, 1].detach().numpy(), color='green', label='Final positions')

# Plot the trajectories of the samples
for i in range(trajectories.shape[1]):
    plt.plot(trajectories[:, i, 0], trajectories[:, i, 1], color='orange', linestyle='--', marker='o', markersize=2)

plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores and Sample Trajectories")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.03
centers_benign = X_train_tensor[y_train == 0]

# Create a mesh grid
xx, yy = torch.meshgrid(torch.linspace(-2, 3, 500), torch.linspace(-1, 2, 500))
grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)

# Compute density estimates for the grid points using the improved KDE function
dens_benign_grid = KDE(grid, centers_benign, bandwidth, kernel='gaussian')
dens_benign_grid = dens_benign_grid.reshape(xx.shape).detach().numpy()

# Select a few samples to track their movement
mals = torch.tensor(X_test[y_test == 1][:10], dtype=torch.float32)

# Perform PGD
x_next, loss_steps_i, trajectories = pgd(mals, X_train_tensor[y_train == 0], 'log', bandwidth, 'gaussian', step_length=0.01, k=100, norm='linf')

# Plot the contour of the KDE score
plt.figure(figsize=(20, 12))
plt.contourf(xx.numpy(), yy.numpy(), dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')

# Plot the initial and final positions of the samples
plt.scatter(mals[:, 0], mals[:, 1], color='red', label='Initial positions')
plt.scatter(x_next[:, 0].detach().numpy(), x_next[:, 1].detach().numpy(), color='green', label='Final positions')

# Plot the trajectories of the samples
for i in range(trajectories.shape[1]):
    plt.plot(trajectories[:, i, 0], trajectories[:, i, 1], color='orange', linestyle='--', marker='o', markersize=2)

plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores and Sample Trajectories")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.4
centers_benign = X_train_tensor[y_train == 0]

# Create a mesh grid
xx, yy = torch.meshgrid(torch.linspace(-2, 3, 500), torch.linspace(-1, 2, 500))
grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)

# Compute density estimates for the grid points using the improved KDE function
dens_benign_grid = KDE(grid, centers_benign, bandwidth, kernel='gaussian')
dens_benign_grid = dens_benign_grid.reshape(xx.shape).detach().numpy()

# Select a few samples to track their movement
mals = torch.tensor(X_test[y_test == 1][:10], dtype=torch.float32)

# Perform PGD
x_next, loss_steps_i, trajectories = pgd(mals, X_train_tensor[y_train == 0], 'log', bandwidth, 'gaussian', step_length=0.01, k=100, norm='linf')

# Plot the contour of the KDE score
plt.figure(figsize=(20, 12))
plt.contourf(xx.numpy(), yy.numpy(), dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')

# Plot the initial and final positions of the samples
plt.scatter(mals[:, 0], mals[:, 1], color='red', label='Initial positions')
plt.scatter(x_next[:, 0].detach().numpy(), x_next[:, 1].detach().numpy(), color='green', label='Final positions')

# Plot the trajectories of the samples
for i in range(trajectories.shape[1]):
    plt.plot(trajectories[:, i, 0], trajectories[:, i, 1], color='orange', linestyle='--', marker='o', markersize=2)

plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores and Sample Trajectories")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

### Step 3 — Choosing the centres

#Centers

In [ ]:
bens = X_train_tensor[y_train == 0]

# Set to track indices of samples to be removed
removed_set = set()

# Step 1: Identify similar samples
for i in range(len(bens)):
    if i in removed_set:
        continue
    sample1 = bens[i]
    for j in range(i + 1, len(bens)):
        if j in removed_set:
            continue
        sample2 = bens[j]
        if torch.abs(sample1 - sample2).sum() < .4:
            removed_set.add(j)

# Step 2: Select samples that are not in removed_set
selected_samples = [sample for idx, sample in enumerate(bens) if idx not in removed_set]

# Step 3: Stack tensors along a new dimension
centers = torch.stack(selected_samples)
print(centers_benign.shape)

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.1
centers_benign = centers

# Create a mesh grid
xx, yy = torch.meshgrid(torch.linspace(-2, 3, 500), torch.linspace(-1, 2, 500))
grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)

# Compute density estimates for the grid points using the improved KDE function
dens_benign_grid = KDE(grid, centers_benign, bandwidth, kernel='gaussian')
dens_benign_grid = dens_benign_grid.reshape(xx.shape).detach().numpy()

# Select a few samples to track their movement
mals = torch.tensor(X_test[y_test == 1][10:20], dtype=torch.float32)

# Perform PGD
x_next, loss_steps_i, trajectories = pgd(mals, centers_benign, 'log', bandwidth, 'gaussian', step_length=0.05, k=30, norm='l2')

# Plot the contour of the KDE score
plt.figure(figsize=(20, 12))
plt.contourf(xx.numpy(), yy.numpy(), dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')

# Plot the initial and final positions of the samples
plt.scatter(mals[:, 0], mals[:, 1], color='red', label='Initial positions')
plt.scatter(x_next[:, 0].detach().numpy(), x_next[:, 1].detach().numpy(), color='green', label='Final positions')

# Plot the trajectories of the samples
for i in range(trajectories.shape[1]):
    plt.plot(trajectories[:, i, 0], trajectories[:, i, 1], color='orange', linestyle='--', marker='o', markersize=2)
for traj in trajectories:
    plt.scatter(traj[:, 0], traj[:, 1], color='green', marker='x', s=10)

plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores and Sample Trajectories")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

In [ ]:
centers2 = X_train_tensor[y_train == 0]

In [ ]:
# Fit KDE model for benign class
bandwidth = 0.2
centers_benign = centers2[:21]

# Create a mesh grid
xx, yy = torch.meshgrid(torch.linspace(-2, 3, 500), torch.linspace(-1, 2, 500))
grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)

# Compute density estimates for the grid points using the improved KDE function
dens_benign_grid = KDE(grid, centers_benign, bandwidth, kernel='gaussian')
dens_benign_grid = dens_benign_grid.reshape(xx.shape).detach().numpy()

# Select a few samples to track their movement
mals = torch.tensor(X_test[y_test == 1][10:20], dtype=torch.float32)

# Perform PGD
x_next, loss_steps_i, trajectories = pgd(mals, centers_benign, 'log', bandwidth, 'gaussian', step_length=0.05, k=30, norm='l2')

# Plot the contour of the KDE score
plt.figure(figsize=(20, 12))
plt.contourf(xx.numpy(), yy.numpy(), dens_benign_grid, levels=20, cmap='Blues')
plt.colorbar(label='Density')

# Plot the initial and final positions of the samples
plt.scatter(mals[:, 0], mals[:, 1], color='red', label='Initial positions')
plt.scatter(x_next[:, 0].detach().numpy(), x_next[:, 1].detach().numpy(), color='green', label='Final positions')

# Plot the trajectories of the samples
for i in range(trajectories.shape[1]):
    plt.plot(trajectories[:, i, 0], trajectories[:, i, 1], color='orange', linestyle='--', marker='o', markersize=2)
for traj in trajectories:
    plt.scatter(traj[:, 0], traj[:, 1], color='green', marker='x', s=10)

plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=20, cmap='coolwarm')
plt.title("Contour Plot of KDE Scores and Sample Trajectories")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## Part 2 — KDE as a detector, under the σ attack

### Step 4 — Setup

# Loading data

In [ ]:
import torch
import torch.nn as nn

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',
                  'https://drive.google.com/uc?id=1sFBZ3A8VQ-iDjmwASg4u5uHQQR8kVJy9',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import zipfile

# Define the path to the zip file and the extraction path
zip_file_path = '/content/datasets.zip'
extract_to_path = '/content/'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 1050
# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
# Function to count samples for each label
def count_labels(labels):
    count_0 = (labels == 0).sum().item()  # Count samples with label = 0
    count_1 = (labels == 1).sum().item()  # Count samples with label = 1
    return count_0, count_1

# Count labels for each dataset
train_count_0, train_count_1 = count_labels(train_labels)
val_count_0, val_count_1 = count_labels(val_labels)
test_count_0, test_count_1 = count_labels(test_labels)

# Calculate total counts across all datasets
total_count_0 = train_count_0 + val_count_0 + test_count_0
total_count_1 = train_count_1 + val_count_1 + test_count_1

# Print results
print(f"Training set             \t Label 0: {train_count_0}\t Label 1: {train_count_1}")
print(f"Validation set           \t Label 0: {val_count_0}\t Label 1: {val_count_1}")
print(f"Test set                 \t Label 0: {test_count_0}\t Label 1: {test_count_1}")
print(f"Total across all datasets\t Label 0: {total_count_0}\t Label 1: {total_count_1}")

In [ ]:
print(f"The avrage number of feature in Training set: {train_features.sum()/train_features.shape[0]:.2f}")
print(f"The avrage number of feature in Validation set: {val_features.sum()/val_features.shape[0]:.2f}")
print(f"The avrage number of feature in Test set: {test_features.sum()/test_features.shape[0]:.2f}")

In [ ]:
removal_array = torch.load('manipulation_x.pt', weights_only=True, map_location=device)
removal_array.sum()

In [ ]:
import torch

def final_rounding(oimgs, newimg, model, removal_array, is_detector_enabled=False, oblivion=False):
    """
    Apply final rounding adjustments to adversarial images based on the model's predictions.

    Parameters:
    - oimgs (Tensor): Original images tensor.
    - newimg (Tensor): Adversarially perturbed images tensor.
    - model (nn.Module): The model used to evaluate active samples.
    - removal_array (Tensor): Array indicating features that should not be modified.

    Returns:
    - final (Tensor): The adjusted adversarial images.
    """

    # Ensure tensors are on the correct device
    device = oimgs.device
    oimgs, perturbed, removal_array = oimgs.to(device), newimg.to(device), removal_array.to(device)

    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Calculate differences and prepare significant change mask
    with torch.no_grad():

        if is_detector_enabled:
            if model.name == 'kde':
                logits, x_hidden = model.forward_f(oimgs)
                prob = model.forward_g(x_hidden, y_pred=0)
            else:
                logits, prob = model.forward(oimgs)
        else:
            logits = model.forward(oimgs)

        if is_detector_enabled and not oblivion:
            tau = model.tau[0].item()
            inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
        else:
            inactive_mask = (logits.argmax(dim=1) == 0)

        active_mask = ~inactive_mask

        # Calculate total change for each sample
        total_change = (perturbed - oimgs).abs()
        significant_mask = total_change > 0.1
        sorted_idx = total_change.argsort(dim=1, descending=True)

        # Define changes to be applied, ensuring correct type and device
        changes = torch.where(perturbed > oimgs, 1, torch.where(perturbed < oimgs, 0, oimgs)).float().to(device)

    # Clone the original images for final adjustments
    final = oimgs.clone()

    # Set a practical maximum for significant changes per sample
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 150)

    # Iterate through each significant change index
    for i in range(max_significant_changes):
        if not active_mask.any():
            break

        active_indices = torch.where(active_mask)[0]

        # Update the final image with changes based on non-fixed feature mask
        indices = sorted_idx[active_indices, i]
        final[active_indices, indices] = torch.where(
            non_fixed_features_mask[active_indices, indices],
            changes[active_indices, indices],
            final[active_indices, indices]
        )

        # Update active_mask based on model prediction after applying changes
        with torch.no_grad():

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final[active_indices])
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final[active_indices])
            else:
                logits = model.forward(final[active_indices])

            if is_detector_enabled and not oblivion:
                inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
            else:
                inactive_mask = (logits.argmax(dim=1) == 0)

            active_mask[active_indices] = (~inactive_mask)

    return final

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")

    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_adv_accuracies(adv_rounded_list, labels, model, mal_x, is_detector_enabled=False, device='cpu', batch_size=512, norm='L2', interval=5, oblivion=True):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(18, 12))
    model.eval()
    mal_x = mal_x.to(device)

    for idx, (adv_rounded, color, label) in tqdm(enumerate(zip(adv_rounded_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if is_detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances)
            sorted_distances, _ = torch.sort(total_distances)
            x_values = sorted_distances.cpu()

            # Calculate percentage values for accuracy on y-axis
            y_values = (1 - torch.arange(1, len(sorted_distances) + 1).float().cpu() / num_samples) * 100

            sns.lineplot(x=x_values, y=y_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    attack_type = 'oblivious attack' if oblivion else 'adaptive attack'

    plt.xlabel(f'Distance ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Model Accuracy (%)', fontsize=14, labelpad=10)
    #plt.title('Model Accuracy vs. Distance from Adversarial Examples', fontsize=16, pad=10)
    plt.title(f'Accuracy (%) of detector {model.name} vs. Distance under {attack_type}', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12, loc='upper right')

    # Set x-axis with integer values but fewer ticks
    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)

    # Y-axis ticks for percentage values, every 10% from 0% to 100%
    plt.yticks(ticks=range(0, 110, 10))

    # Set x-axis ticks at intervals, e.g., every 5 units
    x_min, x_max = int(x_values.min().item()), int(x_values.max().item())
    plt.xticks(ticks=range(x_min, x_max + 1, interval))

    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
from torch.utils.data import DataLoader
from typing import List, Any

def attack_mal_batches(
    test_loader: DataLoader,
    model: nn.Module,
    attack: Any,
    removal_array: torch.Tensor,
    device: torch.device,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    **kwargs
):
    """
    Args:
        test_loader: DataLoader for test data.
        model: Model to be attacked.
        attack: Adversarial attack function (e.g., sigma_zero_rounded).
        removal_array: Masking tensor indicating removable features.
        device: Device for computation.
        is_detector_enabled: Enable/disable detector during attack.
        oblivion: Oblivion flag for attack configuration.
        **kwargs: Additional parameters for the attack function such as steps, lr, etc.

    Returns:
        Tensor with evaluated adversarial examples (rounded).
    """
    #model.to(device)
    model.eval()
    perturbed_batches = []
    mal_x_batches = []

    # Iterate through each batch in the test_loader

    for x_test, y_test in test_loader:
        torch.cuda.empty_cache()
        # Move batch to device
        x_test, y_test = x_test.to(device), y_test.to(device)

        # Select only malicious samples (label == 1)
        mal_x_batch = x_test[y_test.squeeze() == 1]

        if mal_x_batch.size(0) > 0:  # Check if there are any malicious samples
            # Apply the adversarial attack to malicious samples
            perturbed_batch = attack(
                model=model,
                inputs=mal_x_batch,
                removal_array=removal_array,
                device=device,
                is_detector_enabled=is_detector_enabled,
                oblivion=oblivion,
                **kwargs
            )

            # Collect examples
            mal_x_batches.append(mal_x_batch)
            perturbed_batches.append(perturbed_batch)

    # Combine all lists into a single tensor
    combined_mals = torch.cat(mal_x_batches, dim=0)
    combined_advs = torch.cat(perturbed_batches, dim=0)



    return combined_mals, combined_advs

In [ ]:
import torch

def calculate_distance(adv_samples, original_samples, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_samples (torch.Tensor): The adversarial samples.
    - original_samples (torch.Tensor): The original samples.
    - norm (str): The norm type ('l0', 'l1', 'l2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def sum_of_top_n_distances(adv_rounded, mal_x_batch, y_pred, n, norm='L1'):
    """
    Calculate the sum of the top n smallest distances between adv_rounded and mal_x_batch
    for samples where y_pred == 0, based on the specified norm.

    Parameters:
    - adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    - mal_x_batch (torch.Tensor): The original malicious examples tensor.
    - y_pred (torch.Tensor): The prediction tensor.
    - n (int): The number of top distances to sum.
    - norm (str): The norm to use ('l0', 'l1', 'l2').

    Returns:
    - torch.Tensor: The sum of the top n smallest distances.
    """
    distances = calculate_distance(adv_rounded[y_pred == 0], mal_x_batch[y_pred == 0], norm)
    top_n_distances = torch.sort(distances).values[:n]
    return top_n_distances.sum()

def evaluate_adversarial_examples(mal_x, adv, model, removal_array, is_detector_enabled=False, oblivion=False, rounded=False, batch_size=1024, norm='L1'):
    """
    Evaluate the model's performance on adversarial examples, optionally applying rounding, in batches if necessary,
    with support for different norms.

    Parameters:
    - mal_x (Tensor): Original malware input.
    - adv (Tensor): Adversarial examples generated from mal_x.
    - model (nn.Module): The trained model to be evaluated.
    - removal_array (Tensor): Array used for final rounding.
    - rounded (bool, optional): Whether to apply rounding on adversarial examples. Default is True.
    - batch_size (int, optional): The maximum number of samples to process at once. Default is 1024.
    - norm (str): The norm to use for distance calculations ('L0', 'L1', or 'L2').

    Returns:
    - final_adv (Tensor): The (rounded) adversarial examples used in evaluation.
    """
    model.eval()
    torch.cuda.empty_cache()

    num_malwares = mal_x.shape[0]
    num_successful_attacks = 0
    total_distances = 0
    y_pred_list = []
    final_adv_list = []
    all_distances = []

    for i in range(0, num_malwares, batch_size):
        mal_x_batch = mal_x[i:i + batch_size]
        adv_batch = adv[i:i + batch_size]

        # Apply rounding if specified
        final_adv_batch = final_rounding(mal_x_batch, adv_batch, model, removal_array, is_detector_enabled, oblivion) if rounded else adv_batch.clone()
        final_adv_list.append(final_adv_batch)

        # Forward pass and calculate successful attacks
        with torch.no_grad():
            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final_adv_batch)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final_adv_batch)
                #logits, prob = (model.forward_f(final_adv_batch)[0], model.forward_g(model.forward_f(final_adv_batch)[1], y_pred=0)) if model.name == 'kde' else model.forward(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
            else:
                logits = model.forward(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0)

        y_pred_batch = (~successful_attacks_mask).long()
        y_pred_list.append(y_pred_batch)

        # Calculate and accumulate distances for successful attacks
        num_successful_attacks += successful_attacks_mask.sum().item()
        if successful_attacks_mask.any():
            distances = calculate_distance(final_adv_batch[successful_attacks_mask], mal_x_batch[successful_attacks_mask], norm)
            total_distances += distances.sum().item()
            all_distances.extend(distances.cpu().tolist())

    # Final predictions and adversarial examples aggregation
    y_pred = torch.cat(y_pred_list)
    final_adv = torch.cat(final_adv_list)

    # Calculate evaluation metrics
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100
    mean_distance = total_distances / max(1, num_successful_attacks)
    median_distance = torch.median(torch.tensor(all_distances)).item() if all_distances else 0

    # Print evaluation results
    print(f"Evaluation Results on Adversarial Examples based on Distance:({norm} norm), Rounded:{rounded}")
    print("-----------------------------------------------")
    print(f"Number of successful attacks: {num_successful_attacks}/{num_malwares}")
    print(f"Model accuracy on malware under attack: {accuracy:.2f}%")
    print(f"Mean distance (successful adv. examples vs. original): {mean_distance:.2f}")
    print(f"Median distance (successful adv. examples vs. original): {median_distance:.2f}")

    # Calculate and print mean distances for various TOP N thresholds
    top_n_values = [500, 1000, 2000, 2500, 3000]
    for top_n in top_n_values:
        if num_malwares >= top_n:
            total_distance_top_n = sum_of_top_n_distances(final_adv, mal_x, y_pred, top_n, norm=norm) / top_n
            print(f"Mean distance (successful adv. examples vs. original) - TOP {top_n}: {total_distance_top_n:.2f}")

    return final_adv

### Step 5 — Load and check the KDE detector

# Loadin KDE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dropout = 0.6, alpha = 0.2, smooth = False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences)
        gt_labels = torch.cat(gt_labels, dim=0)
        return confidences, gt_labels


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

        if np.any([np.all(y_true == i) for i in range(self.n_classes)]):
            print("class absent.")
            return

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")


    def load(self, model_save_path):
        """
        load model parameters from disk
        """
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

In [ ]:
model_dnn

In [ ]:
model_dnn.double().to(device)
model_dnn.predict(test_loader)

In [ ]:
model_dnn.float()

In [ ]:
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class KernelDensityEstimation(nn.Module, DetectorTemplate):
    """
    kernel density estimation upon the penultimate layer

    parameters
    -------------
    @param model, torch.nn.Module, an instantiation of model object
    @param bandwidth, float, variance of Gaussian density function
    @param n_classes, integer, number of classes
    @param ratio, float [0,1], ratio for computing the threshold
    """

    def __init__(self, model, n_centers=1000, bandwidth=20., n_classes=2, ratio=0.95, name='KDE', device='cpu'):
        super(KernelDensityEstimation, self).__init__()
        assert isinstance(model, torch.nn.Module)
        self.model = model
        self.device = device
        self.name = name
        self.n_centers = n_centers
        self.bandwidth = bandwidth
        self.n_classes = n_classes
        self.ratio = ratio
        self.gaussian_means = None

        self.tau = nn.Parameter(torch.zeros([self.n_classes, ], device=self.device), requires_grad=False)

    def forward(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        x_prob = self.forward_g(x, logits.argmax(1).detach())
        return logits, x_prob

    def forward_f(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        return logits, x

    def forward_g(self, x_hidden, y_pred):
        """
        1 / |X_| * \sum(exp(||x - x_||_2^2/sigma^2))

        parameters
        -----------
        @param x_hidden, torch.tensor, node hidden representation
        @param y_pred, torch.tensor, prediction
        """
        size = x_hidden.size()[0]
        dist = [torch.sum(torch.square(means.unsqueeze(dim=0) - x_hidden.unsqueeze(dim=1)), dim=-1) for means in
                self.gaussian_means]
        kd = torch.stack([torch.mean(torch.exp(-d / self.bandwidth ** 2), dim=-1) for d in dist], dim=1)
        # return p(x|y=y_pred)
        return -1 * kd[torch.arange(size), y_pred]

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for density estimation
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        self.eval()
        probabilities = []
        gt_labels = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                logits, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
                gt_labels.append(y_val)
            prob = torch.cat(probabilities, dim=0)
            gt_labels = torch.cat(gt_labels)
            for i in range(self.n_classes):
                prob_x_y = prob[gt_labels == i]
                s, _ = torch.sort(prob_x_y)
                self.tau[i] = s[int((s.shape[0] - 1) * ratio)]

    def predict(self, test_data_producer, indicator_masking=True):
        # evaluation on detector & indicator
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob, y_pred).cpu().numpy()
        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.

        print('The indicator is turning on...')
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"Number of considered samples is {len(y_true)}%")
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr:.5f}%, False Positive Rate (FPR) is {fpr:.5f}%, F1 score is {f1:.5f}%")


    def eval(self):
        self.model.eval()

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = x.to(device), y.to(device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(F.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)

        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau[y_pred]

    def indicator(self, x_prob, y_pred=None):
        assert y_pred is not None
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.get_tau_sample_wise(y_pred)).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.get_tau_sample_wise(y_pred)
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")
        # res = probability.reshape(-1, 1).repeat_interleave(2, dim=1) >= self.tau
        # return res[torch.arange(res.size()[0]), y_pred]


    def load(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.gaussian_means = ckpt['gaussian_means']
        self.tau = ckpt['tau']
        self.model.load_state_dict(ckpt['base_model'])

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_kde = KernelDensityEstimation(model_dnn,n_centers=1000,bandwidth=16.,n_classes=2,ratio=0.95,device=device,name='KDE')
model_kde.load('/content/amd_kde.pth')
model_kde.to(device).float()

## Check KDE

In [ ]:
model_kde.predict(test_loader, indicator_masking=True)

In [ ]:
model_kde.predict(test_loader, indicator_masking=False)

In [ ]:
model_kde.tau

In [ ]:
for x_test, y_test in train_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

In [ ]:
logits, x_prob = model_kde.forward(mal_x_batch[:10])
print(logits, x_prob)

In [ ]:
(x_prob <model_kde.tau[1]).sum()

In [ ]:
model_kde((mal_x_batch[:5]+0.01))

In [ ]:
model_kde((mal_x_batch[:5]+0.05))

### Step 6 — The σ attack on KDE, and the oblivion variant

# KDE - sigma_zero

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples

def sigma_zero(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 4,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        print(f"{'-'*50}")
        print(f"Outer step: {outer_step + 1}")
        print("unique values of CONST :", torch.unique(CONST))
        active_inputs = inputs.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100)

        # Dynamic threshold per feature, and success masks
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)
        success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score, mal_score = logits[:, 0], logits[:, 1]

            # Calculate decision loss
            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + \
                          CONST * torch.clamp(prob - tau + confidence2, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)

            # Success mask and update best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)

                true_l0 = delta.data.ne(0).sum(dim=1)  # ne(0) (short for "not equal to zero")
                improved_mask = success_mask & (true_l0 < best_l0)
                best_l0= torch.where(improved_mask, true_l0, best_l0)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)

                success_samples |= success_mask
                o_success_samples |= success_mask

            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 500 == 0):
                avg_best_dist = best_l0[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Distance (avg): {avg_best_dist:6.2f} | "
                    f"Loss: {adv_loss.item():7.2f}"
                )

            # Optimize delta
            optimizer.zero_grad()
            adv_loss.backward()

            # Gradient normalization for stability
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)

            # Update delta and apply scheduler
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Adjust CONST based on success and updated bounds after the first outer step
        if (outer_step + 1) !=  binary_search_steps:
            upper_bound[success_samples] = torch.minimum(upper_bound[success_samples], CONST[success_samples])
            lower_bound[~success_samples] = torch.maximum(lower_bound[~success_samples], CONST[~success_samples])

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound < 1e8
            updated_lower_limit_mask = lower_bound > 0

            CONST[success_samples] /= 100.0
            CONST[~success_samples] *= 100.0

            # Set CONST for active samples that have both updated lower and upper bounds
            CONST[success_samples & updated_lower_limit_mask] = (
                (lower_bound[success_samples & updated_lower_limit_mask] +
                 upper_bound[success_samples & updated_lower_limit_mask]) / 2.
            )
            CONST[~success_samples & updated_upper_limit_mask] = (
                (lower_bound[~success_samples & updated_upper_limit_mask] +
                 upper_bound[~success_samples & updated_upper_limit_mask]) / 2.
            )


    return inputs + best_delta

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

def sigma_zero_rounded(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Set outer loop iterations
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        print(f"{'-'*50}")
        print(f"Outer step: {outer_step + 1}")
        print("unique values of CONST :", torch.unique(CONST))
        active_inputs = inputs.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Set up confidence, thresholds, and success masks
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples
            adv_inputs = active_inputs + delta

            # Forward pass
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score, mal_score = logits[:, 0], logits[:, 1]

            # Calculate decision loss
            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + \
                          CONST * torch.clamp(prob - tau + confidence2, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)

            # Success masks and updating best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(active_inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)

                    if is_detector_enabled:
                        if model.name == 'KDE':
                            rounded_logits, x_hidden = model.forward_f(adv_inputs_rounded)
                            rounded_prob = model.forward_g(x_hidden, y_pred=0)
                        else:
                            rounded_logits, rounded_prob = model.forward(adv_inputs_rounded)
                    else:
                        rounded_logits = model.forward(adv_inputs_rounded)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)

                    dist = (adv_inputs_rounded != active_inputs).float().sum(dim=1)
                    #dist = torch.abs(active_inputs - adv_inputs_rounded).sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)

                    success_samples |= success_mask
                    o_success_samples |= success_mask
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    confidence[success_mask_R & success_samples] -= 0.1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence[success_mask & ~success_mask_R] += 0.1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence[~success_mask & ~success_mask_R] = 1e-4

                    confidence2[success_mask_R & success_samples] -= 0.1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence2[success_mask & ~success_mask_R] += 0.1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence2[~success_mask & ~success_mask_R] = 1e-4

            # Calculate L0 approximation and combined loss
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose logging
            if verbose and (step == 0 or (step + 1) % 500 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f}"
                )

            # Backpropagation and optimization
            optimizer.zero_grad()
            adv_loss.backward()

            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)

            optimizer.step()
            scheduler.step()

            # Enforce constraints on delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Update bounds after outer step
        if (outer_step + 1) != binary_search_steps:
            upper_bound[success_samples] = torch.minimum(upper_bound[success_samples], CONST[success_samples])
            lower_bound[~success_samples] = torch.maximum(lower_bound[~success_samples], CONST[~success_samples])

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound < 1e8
            updated_lower_limit_mask = lower_bound > 0

            CONST[success_samples] /= 100.0
            CONST[~success_samples] *= 100.0

            # Set CONST for active samples that have both updated lower and upper bounds
            CONST[success_samples & updated_lower_limit_mask] = (
                (lower_bound[success_samples & updated_lower_limit_mask] +
                 upper_bound[success_samples & updated_lower_limit_mask]) / 2.
            )
            CONST[~success_samples & updated_upper_limit_mask] = (
                (lower_bound[~success_samples & updated_upper_limit_mask] +
                 upper_bound[~success_samples & updated_upper_limit_mask]) / 2.
            )

    return inputs + best_delta

In [ ]:
learning_rates = [0.6, 0.4, 0.2]  # Define the different learning rates you want to test

for lr in learning_rates:
    print(f"\nTesting with learning rate: {lr}")
    combined_mals, combined_advs = attack_mal_batches(
        test_loader=test_loader,
        model=model_kde,
        attack=sigma_zero,
        removal_array=removal_array,
        device=device,
        is_detector_enabled=True,
        oblivion=False,
        steps=1000,
        lr=lr,  # Use current learning rate from the list
        sigma=1e-4,
        threshold=0.3,
        verbose=True,
        grad_norm=torch.inf,
        binary_search_steps=4,
        initial_const=0.1
    )

    print(f"\nTesting with learning rate: {lr}")
    result = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_kde, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size,norm='L0')

In [ ]:
combined_mals, combined_advs = attack_mal_batches(
    test_loader=test_loader,
    model=model_kde,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.5,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=4,
    initial_const=0.1
)

In [ ]:
ensure_fixed_elements_unchanged(combined_mals, combined_advs, removal_array)

In [ ]:
result = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_kde, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size,norm='L0')

result2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_kde, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
combined_mals, combined_advs_round = attack_mal_batches(
    test_loader=test_loader,
    model=model_kde,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.5,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=4,
    initial_const=0.1
)

In [ ]:
ensure_fixed_elements_unchanged(combined_mals, combined_advs_round, removal_array)

In [ ]:
result_rounded = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=model_kde, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size, norm='L0')

result_rounded2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=model_kde, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
result_rounded3 = evaluate_adversarial_examples(mal_x=combined_mals, adv=result_rounded2, model=model_kde, removal_array=removal_array,
                                              is_detector_enabled=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result,result2,result_rounded3]
labels = ['sigma_zero','sigma_zero_r','sigma_zero_rounded']
plot_adv_accuracies(adv_rounded_list, labels, model_kde, combined_mals, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=2, oblivion=False)

## oblivion=True

In [ ]:
# Usage example with specific parameters passed via **kwargs
combined_mals, combined_advs = attack_mal_batches(
    test_loader=test_loader,
    model=model_kde,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.5,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=4,
    initial_const=0.1
)

In [ ]:
result = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_kde, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

result2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_kde, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
# Usage example with specific parameters passed via **kwargs
combined_mals, combined_advs_round = attack_mal_batches(
    test_loader=test_loader,
    model=model_kde,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.5,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=4,
    initial_const=0.1
)

In [ ]:
result_rounded = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=model_kde, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

result_rounded2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=model_kde, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
result_rounded3 = evaluate_adversarial_examples(mal_x=combined_mals, adv=result_rounded2, model=model_kde, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result,result2,result_rounded3]
labels = ['sigma_zero','sigma_zero_r','sigma_zero_rounded']
plot_adv_accuracies(adv_rounded_list, labels, model_kde, combined_mals, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=2, oblivion=True)

## Part 3 — GKDE: the density term as an attack (from `kde1.ipynb`)

### Step 7 — Setup: the models and the KDE centres

In [ ]:
#RBF_models

download_links = [
                  'https://drive.google.com/uc?id=1-8lJXLdAl_4NdDwzw9kFML0aiOCTrI9f',
                  'https://drive.google.com/uc?id=1-OHACrNCt0yKBbdqQPVfNZcjKt5_jxKD',
                  'https://drive.google.com/uc?id=1-KeXJXtU1_6m9JOhormeVwigy0myX3HL',
                  'https://drive.google.com/uc?id=1-13RDdZqnrNkdHg3D8PC5KI0CZREwlsz',
                  'https://drive.google.com/uc?id=1-8LjsCdzKh6asxCFsYLiQZbSEXXKSQBP',

]

import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM_weightedLoss.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py'
]

In [ ]:
import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from adverserial_attacks_functions import *


torch.manual_seed(0)

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    #os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
model_DNN = MalwareDetectionModel().to(device)
# Load model parameters
model_DNN.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM_weightedLoss = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM_weightedLoss.load_state_dict(torch.load('model_AT_rFGSM_weightedLoss.pth', map_location=torch.device(device)))

In [ ]:
model_evaluation(model_DNN, test_loader, device)

In [ ]:
model_evaluation(model_AT_rFGSM,test_loader,device)

In [ ]:
model_evaluation(model_AT_rFGSM_weightedLoss,test_loader,device)

In [ ]:
class RBFModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, init_centers, init_sigmas, kernel):
        super(RBFModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.centers = nn.Parameter(torch.Tensor(init_centers))
        self.sigmas = nn.Parameter(torch.Tensor(init_sigmas))
        self.kernel = kernel
        # Linear layer for output
        self.linear = nn.Linear(hidden_dim, output_dim)

    def gaussian(self, x, c, sigma):
        return torch.exp(-torch.sum((x[:, None, :] - c) ** 2, dim=-1) / (2 * sigma ** 2))

    def laplacian(self, x, c, sigma):
        return torch.exp(-torch.sum(torch.abs(x[:, None, :] - c) , dim=-1) / sigma)

    def forward(self, x):
      if self.kernel == 'gaussian':
        radial_out = self.gaussian(x, self.centers, self.sigmas)
      elif self.kernel == 'laplacian':
        radial_out = self.laplacian(x, self.centers, self.sigmas)
      else:
        raise ValueError("Invalid kernel type. Choose 'gaussian' or 'laplacian'.")

      output = self.linear(radial_out.to(torch.float32))
      return output

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 10000))
model_gaussian_1000 = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000 = model_gaussian_1000.to(device)

# Load the model state dictionary
model_gaussian_1000.load_state_dict(torch.load('/content/best_model_gaussian_1000.pth', map_location=torch.device(device)))

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 1144))
model_gaussian_1000_nonremoval = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000_nonremoval = model_gaussian_1000_nonremoval.to(device)

# Load the model state dictionary
model_gaussian_1000_nonremoval.load_state_dict(torch.load('/content/best_model_gaussian_1000_nonremoval.pth', map_location=torch.device(device)))

# New Section

centers

In [ ]:
# Pre-select benign samples
benign_samples = []
for x_batch, y_batch in test_loader:
  benign_samples.append(x_batch[y_batch.squeeze() == 0])

ben_x = torch.cat(benign_samples, dim=0).to(device)

# Pre-select benign samples
mal_samples = []
for x_batch, y_batch in test_loader:
  mal_samples.append(x_batch[y_batch.squeeze() == 1])

mal_x = torch.cat(mal_samples, dim=0).to(device)

print(ben_x.shape)
print(mal_x.shape)

In [ ]:
num_centers = 1000

# Set the random seed for reproducibility
torch.manual_seed(42)

# Generate a random permutation of indices
indices = torch.randperm(ben_x.size(0))

# Select the first `num_samples_to_select` indices
selected_indices = indices[:num_centers]

# Use the selected indices to index into the original tensor
centers = ben_x[selected_indices].to(device)

centers.shape

In [ ]:
def KDE(x, centers, bandwidth=10.):

    u = torch.abs(x[:, None, :] - centers)


    # Compute kernel contributions for all data points
    kernel_contributions = torch.exp(-torch.sum(u, dim=-1) / bandwidth)
    #kernel_contributions = torch.exp(-torch.sum(u**2, dim=-1) / bandwidth**2)
    # Sum contributions across all data points
    estimate = torch.mean(kernel_contributions, dim=1)

    return estimate

In [ ]:
KDE(ben_x[:10], centers, bandwidth=10.)

In [ ]:
KDE(mal_x[:10], centers, bandwidth=10.)

### Step 8 — PGD with and without the KDE term

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd(x, y, model, centers, bandwidth, penalty_kde, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5,
        round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Boundary calculations
    scaled_max_extended = torch.maximum(insertion_array.float(), x)
    scaled_min_extended = torch.minimum((1. - removal_array.float()), x)

    # Multi-step PGD
    for t in range(k):

        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        ce = criterion(model(x_var), y.view(-1).long())
        kde = KDE(x_var, centers, bandwidth)
        loss = ce + (penalty_kde * kde)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=scaled_min_extended, max=scaled_max_extended)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of updated features',torch.mean(diff))

    return x_next

In [ ]:
pgd(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM,centers.to(device),10., 10., insertion_array, removal_array, k=50, step_length=1., norm='l2',initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
#without kde
attack_params =  {'centers':centers.to(device),'bandwidth':10.,'penalty_kde':0,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'centers':centers.to(device),'bandwidth':10.,'penalty_kde':10,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

### Step 9 — The paper's parameters, and the penalty-factor grid

**paper** , k=1000, step=0.01

```
For GDKDE [6], PGD-Adam [23],
and `∞ norm-based PGD attack (dubbed PGD-`∞), we iterate
the algorithm with the maximum iterations 1,000 and step size
0.01. The `2 norm-based PGD attack (dubbed PGD-`2) is set
with the maximum iterations 1,000 and step size 1.0.
```


**github**, k=1000, step=1

```
'gdkde' : {'step_size': 1.,'max_iteration': 1000,'negative_data_num': 5000,'kernel_width': 20.,
               'lambda_factor': 1.,'distance_max': 20., 'xi': 0., 'batch_size': 50},
```


In [ ]:
attack_params =  {'centers':centers.to(device),'bandwidth':20.,'penalty_kde':1,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 1000, 'step_length': 0.01, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'centers':centers.to(device),'bandwidth':20.,'penalty_kde':1,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 500, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
def pgd_min2_kde(x, y, model,centers, bandwidth, penalty_factor, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long())) -  penalty_factor * KDE(x, centers, bandwidth)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        loss = criterion(model(x_var), torch.zeros_like(y.view(-1).long())) -  penalty_factor * KDE(x_var, centers, bandwidth)
        #loss_steps.append(loss.mean().detach().item())
        loss_steps_d.append(criterion(model(x_var), y.view(-1).long()).mean().detach().item())
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
bandwidth = 20.
penalty_factor = 10.
attack_params =  {'centers':centers.to(device),'bandwidth': bandwidth,'penalty_factor': penalty_factor,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_kde, device, **attack_params)

In [ ]:
bandwidth = 20.
penalty_factor = 1.
attack_params =  {'centers':centers.to(device),'bandwidth': bandwidth,'penalty_factor': penalty_factor,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_kde, device, **attack_params)

In [ ]:
bandwidth = 20.
penalty_factor = 0.
attack_params =  {'centers':centers.to(device),'bandwidth': bandwidth,'penalty_factor': penalty_factor,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_kde, device, **attack_params)